<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/experiment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 3 — The Engineering Comparison (RQ3)

**Question:** Does the focal-loss-engineered O-XGBoost improve scam detection over standard XGBoost, and under what conditions?

- **Experiment 3:** standard XGBoost vs O-XGBoost at the default focal-loss setting.
- **Experiment 3b:** tuning the focal-loss focusing parameter γ (2 → 3 → 5).

All trained on the localised (UCI + weighted Ghana) data and evaluated on the Ghanaian test set.

In [1]:
!pip install -U xgboost -q

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import recall_score, precision_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
print("Ready. Seed =", SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 5.3 MB/s eta 0:00:00
Ready. Seed = 42


In [2]:
# Public UCI (Western) data
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])
uci['label'] = uci['label'].map({'ham': 0, 'spam': 1})
uci['source'] = 'uci'

# Ghanaian field data (upload ghana_momo_field.csv to the session first)
field = pd.read_csv('ghana_momo_field.csv')[['label', 'message']]
field['source'] = 'field'

print("UCI:", len(uci), "| Field:", len(field))

UCI: 5572 | Field: 208


In [3]:
# Split field data 60/40. The SAME test set is used everywhere for comparability.
field_train, field_test = train_test_split(
    field, test_size=0.4, stratify=field['label'], random_state=SEED
)
print("Field train:", len(field_train), "| Field test:", len(field_test))
print("Test scam/legit:", dict(field_test['label'].value_counts()))

Field train: 124 | Field test: 84
Test scam/legit: {0: np.int64(67), 1: np.int64(17)}


In [4]:
# Localised training set (UCI + Ghanaian field training portion)
train_B = pd.concat([uci, field_train], ignore_index=True)

vec = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
X_train = vec.fit_transform(train_B['message']); y_train = train_B['label']
X_test = vec.transform(field_test['message']);   y_test = field_test['label']

weights = np.where(train_B['source'] == 'field', 6, 1)
dtrain = xgb.DMatrix(X_train, label=y_train, weight=weights)
dtest  = xgb.DMatrix(X_test, label=y_test)
params = {'max_depth':6,'eta':0.1,'subsample':0.8,'colsample_bytree':0.8,'seed':SEED}
print("Localised training data ready.")

Localised training data ready.


In [5]:
# Adjustable focal loss
def make_focal(gamma, alpha=0.25):
    def obj(y_pred, dtrain):
        y = dtrain.get_label()
        p = 1.0/(1.0+np.exp(-y_pred))
        p_t = y*p + (1-y)*(1-p)
        a_t = y*alpha + (1-y)*(1-alpha)
        fw = a_t*np.power(1-p_t, gamma)
        return fw*(p-y), fw*p*(1-p)*(gamma*(1-p_t)+1)
    return obj
print("Focal loss defined.")

Focal loss defined.


### Experiment 3 — default setting (standard vs O-XGBoost)

In [6]:
std_model = xgb.train({**params,'objective':'binary:logistic'}, dtrain, 200)
o_model   = xgb.train(params, dtrain, 200, obj=make_focal(2.0))

for m, name in [(std_model,"Standard XGBoost"), (o_model,"O-XGBoost (default gamma=2)")]:
    probs = 1.0/(1.0+np.exp(-m.predict(dtest)))
    preds = (probs>=0.5).astype(int)
    print(f"\n=== {name} ===")
    print("Scam recall:", round(recall_score(y_test,preds,pos_label=1)*100,1),"%")
    print(confusion_matrix(y_test,preds))


=== Standard XGBoost ===
Scam recall: 100.0 %
[[ 0 67]
 [ 0 17]]

=== O-XGBoost (default gamma=2) ===
Scam recall: 5.9 %
[[60  7]
 [16  1]]


### Experiment 3b — tuning the focal-loss parameter γ

In [7]:
print("gamma | recall | precision | scams caught")
print("-"*45)
for g in [2, 3, 5]:
    m = xgb.train(params, dtrain, 200, obj=make_focal(g))
    probs = 1.0/(1.0+np.exp(-m.predict(dtest)))
    preds = (probs>=0.3).astype(int)
    r = recall_score(y_test, preds, pos_label=1, zero_division=0)
    p = precision_score(y_test, preds, pos_label=1, zero_division=0)
    caught = int(((preds==1)&(y_test==1)).sum()); tot = int((y_test==1).sum())
    print(f"  {g}   | {r*100:5.1f}% |  {p*100:5.1f}%  |   {caught}/{tot}")

gamma | recall | precision | scams caught
---------------------------------------------
  2   |  52.9% |   33.3%  |   9/17
  3   |  70.6% |   27.3%  |   12/17
  5   |  94.1% |   20.0%  |   16/17
